In [18]:
import json, sys, gzip
from pathlib import Path

def lines(p):
    p = Path(p)
    openf = gzip.open if p.suffix == ".gz" else open
    with openf(p, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield line

def suggest_out_path(in_path: str, out_path: str | None = None) -> str:
    if out_path:
        return out_path
    p = Path(in_path)
    name = p.name
    if name.endswith(".jsonl.gz"):
        out_name = name.replace(".jsonl.gz", ".correct.jsonl.gz")
    elif name.endswith(".jsonl"):
        out_name = name.replace(".jsonl", ".correct.jsonl")
    else:
        out_name = name + ".correct.jsonl"
    return str(p.with_name(out_name))

def open_writer(path: str):
    return gzip.open(path, "wt", encoding="utf-8") if path.endswith(".gz") else open(path, "w", encoding="utf-8")

def extract_prompt(obj: dict) -> str | None:
    # Prefer explicit prompt_text; fallback to the first user message content; else join all contents.
    if obj.get("prompt_text"):
        return obj["prompt_text"]
    msgs = obj.get("messages")
    if isinstance(msgs, list) and msgs:
        # try first user message
        for m in msgs:
            if isinstance(m, dict) and m.get("role") == "user" and "content" in m:
                return m["content"]
        # fallback: join all message contents
        try:
            return "\n\n".join([m.get("content", "") for m in msgs if isinstance(m, dict)])
        except Exception:
            return None
    return None

def main(in_path: str, out_path: str | None = None, verbose: bool = False):
    out_path = suggest_out_path(in_path, out_path)
    total = true_cnt = miss = bad = 0

    with open_writer(out_path) as wf:
        for ln in lines(in_path):
            total += 1
            try:
                obj = json.loads(ln)
            except json.JSONDecodeError:
                bad += 1
                continue

            v = obj.get("ids_match")
            if v is None:
                miss += 1

            if v is True:
                true_cnt += 1
                if verbose and obj.get("model_reply"):
                    print(obj["model_reply"])

                # Build a compact record with required metadata + a few useful fields
                rec = {
                    "task": obj.get("task"),
                    "split": obj.get("split"),
                    "k": obj.get("k"),
                    "sample_idx": obj.get("sample_idx"),
                    "index": obj.get("index"),
                    "data_source": obj.get("data_source"),

                    "ground_truth": obj.get("ground_truth"),
                    "pred_ids": obj.get("pred_ids"),
                    "pred_answer": obj.get("pred_answer"),
                    "answer_match": obj.get("answer_match"),
                    "ids_match": obj.get("ids_match"),
                    "model_reply": obj.get("model_reply"),

                    # Normalized prompt field
                    "prompt": extract_prompt(obj),
                }
                wf.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"wrote: {out_path}")
    print(f"ids_match true: {true_cnt}")
    print(f"total rows: {total}")
    if total:
        print(true_cnt, total)
        print(f"percent: {100 * true_cnt/total:.4f}")
    if miss:
        print(f"(missing ids_match: {miss})")
    if bad:
        print(f"(malformed JSON lines skipped: {bad})")


In [19]:
in_path = 'passk_out/lis_Qwen2.5-7B-Instruct_predictions_passk.jsonl'
out_path = 'analysis/lis_Qwen2.5-7B-Instruct_predictions_passk.jsonl'
main(in_path, out_path, False)

wrote: analysis/lis_Qwen2.5-7B-Instruct_predictions_passk.jsonl
ids_match true: 5579
total rows: 109568
5579 109568
percent: 5.0918


In [20]:
5579/109568*100

5.091815128504673

In [21]:
109568/256

428.0